<a href="https://colab.research.google.com/github/daffu081/Employee-Attrition-Analysis/blob/main/chromadb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files

uploaded = files.upload()

Saving innotrans_2026_floor_plan_verified_v7.json to innotrans_2026_floor_plan_verified_v7 (1).json
Saving exhibitors.json to exhibitors.json


In [3]:
!pip install pandas openpyxl tqdm -q

In [16]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   InnoTrans 2026 — ChromaDB Dataset Builder                             ║
# ║   Run this in Google Colab (Python 3.10+)                               ║
# ║                                                                          ║
# ║   Required uploads:                                                      ║
# ║     • innotrans_2026_floor_plan_verified_v7.json                        ║
# ║     • exhibitors.json                                                    ║
# ║                                                                          ║
# ║   Output:                                                                ║
# ║     • /content/chromadb_innotrans/   (persistent ChromaDB)              ║
# ║     • /content/chromadb_innotrans.zip (downloadable archive)            ║
# ╚══════════════════════════════════════════════════════════════════════════╝


# ── CELL 1: Install dependencies ─────────────────────────────────────────────
# Run this cell first, then restart the runtime if prompted.

# !pip install chromadb sentence-transformers


# ── CELL 2: Upload your JSON files ───────────────────────────────────────────

# from google.colab import files
# print("Upload both JSON files:")
# uploaded = files.upload()
# # After upload, files land at /content/<filename>


# ── CELL 3: Imports & configuration ──────────────────────────────────────────

import json
import hashlib
import shutil
import os
import math

import chromadb
from chromadb import EmbeddingFunction, Documents, Embeddings
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# ── Paths — adjust if your files are named differently ───────────────────────
FLOOR_PLAN_PATH = "/content/innotrans_2026_floor_plan_verified_v7.json"
EXHIBITORS_PATH = "/content/exhibitors.json"
DB_PATH         = "/content/chromadb_innotrans"

# ── CELL 4: Choose embedding function ────────────────────────────────────────
#
#   Option A (recommended in Colab — real semantic embeddings):
#     Uses all-MiniLM-L6-v2, downloads ~90 MB on first run.
#
#   Option B (offline / no download — deterministic hash placeholder):
#     Fast, no internet needed, but similarity search is not meaningful.
#     Good for schema validation and pipeline testing.
#
# Set USE_REAL_EMBEDDINGS = True for Option A, False for Option B.

USE_REAL_EMBEDDINGS = True  # ← change to False if you want the hash fallback


class DeterministicHashEF(EmbeddingFunction):
    """
    Offline placeholder: deterministic 384-dim unit-normalized embedding.
    Each dimension is SHA-256(text + dim_index). No NaN/Inf values.
    Replace with a real model for production similarity search.
    """
    DIM = 384

    def __init__(self):
        pass

    def __call__(self, input: Documents) -> Embeddings:
        result = []
        for text in input:
            seed = text.encode()
            floats = []
            for i in range(self.DIM):
                h = hashlib.sha256(seed + i.to_bytes(4, "big")).digest()
                val = int.from_bytes(h[:4], "big", signed=True) / (2 ** 31)
                floats.append(val)
            norm = math.sqrt(sum(x * x for x in floats)) or 1.0
            result.append([x / norm for x in floats])
        return result


if USE_REAL_EMBEDDINGS:
    print("Loading all-MiniLM-L6-v2 (downloads ~90 MB on first run)…")
    ef = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
    print("Embedding model ready.")
else:
    print("Using deterministic hash embeddings (offline placeholder).")
    ef = DeterministicHashEF()


# ── CELL 5: Helper utilities ──────────────────────────────────────────────────

def s(v):
    """
    Sanitize a value for ChromaDB metadata.
    ChromaDB 1.x rejects None — convert to empty string.
    All other non-scalar types are stringified.
    """
    if v is None:
        return ""
    if isinstance(v, (str, int, float, bool)):
        return v
    return str(v)


def stable_id(prefix: str, text: str) -> str:
    """Stable, collision-resistant ID from a prefix + content string."""
    return f"{prefix}_{hashlib.md5(text.encode()).hexdigest()[:10]}"


# ── CELL 6: Load source JSON files ───────────────────────────────────────────

print("Loading floor plan JSON…")
with open(FLOOR_PLAN_PATH, encoding="utf-8") as f:
    fp = json.load(f)

print("Loading exhibitors JSON…")
with open(EXHIBITORS_PATH, encoding="utf-8") as f:
    exhibitors = json.load(f)

event_info = fp["event"]
venue_info = fp["venue"]

print(f"Event  : {event_info['name']}  ({event_info['dates_display']})")
print(f"Venue  : {venue_info['name']}")
print(f"Halls  : {len(fp['halls'])}")
print(f"Exhibitors: {len(exhibitors)}")


# ── CELL 7: Initialise ChromaDB ───────────────────────────────────────────────

if os.path.exists(DB_PATH):
    shutil.rmtree(DB_PATH)
    print(f"Removed existing DB at {DB_PATH}")

client = chromadb.PersistentClient(path=DB_PATH)
print(f"ChromaDB initialised at {DB_PATH}")


# ═════════════════════════════════════════════════════════════════════════════
# COLLECTION 1 — floor_plan_halls
#
# What it stores:
#   One document per hall/sub-hall at Messe Berlin.
#   Includes building, level, venue zone, exhibition themes,
#   and internal zone letters (e.g. 7.1a, 7.1b, 7.1c).
#
# Covers intents (from XLSX):
#   booth_location | hall_navigation | exhibitor_listing |
#   nearby_context | booth_zone
# ═════════════════════════════════════════════════════════════════════════════

print("\n[1/5] Building floor_plan_halls…")

col_halls = client.create_collection("floor_plan_halls", embedding_function=ef)

hall_docs, hall_metas, hall_ids = [], [], []

# Pre-build lookup: hall_id → internal zone labels (e.g. ["a","b","c"])
zone_lookup = {}
for z in fp["hall_internal_zones"].get("zones_by_hall_or_hall_group", []):
    zone_lookup[z["id"]] = z.get("visible_zone_labels", [])

for hall in fp["halls"]:
    hid      = s(hall["id"])
    label    = s(hall.get("label", f"Hall {hid}"))
    building = s(hall.get("building", ""))
    level    = s(hall.get("level", ""))
    vzone    = s(hall.get("venue_zone", ""))
    themes   = hall.get("themes") or []
    t_cands  = hall.get("theme_candidates") or []
    zones    = zone_lookup.get(hid, [])

    zone_text = (
        f" Internal zone sections: {', '.join(zones)}." if zones else ""
    )
    theme_text = (
        f" Exhibition themes: {', '.join(themes)}." if themes
        else (
            f" Likely themes (estimate only): {', '.join(t_cands)}."
            if t_cands else ""
        )
    )

    doc = (
        f"{label} is in Building {building}, Level {level} "
        f"at {venue_info['name']} "
        f"({event_info['name']}, Berlin, Germany, {event_info['dates_display']})."
        f"{theme_text}{zone_text} Venue zone: {vzone}."
    )

    hall_docs.append(doc)
    hall_metas.append({
        "intent":            "booth_location|hall_navigation|exhibitor_listing|nearby_context|booth_zone",
        "json_source":       "Floor Plan",
        "entity_type":       "hall",
        "hall_id":           hid,
        "label":             label,
        "building":          building,
        "level":             level,
        "venue_zone":        vzone,
        "has_internal_zones": "true" if zones else "false",
    })
    hall_ids.append(f"hall_{hid}")

col_halls.add(documents=hall_docs, metadatas=hall_metas, ids=hall_ids)
print(f"    ✓ {col_halls.count()} documents added")


# ═════════════════════════════════════════════════════════════════════════════
# COLLECTION 2 — floor_plan_navigation
#
# What it stores:
#   Entrances, gates, buildings/structures, spatial relationships,
#   and roads & streets around the venue.
#
# Covers intents:
#   booth_navigation | hall_navigation | landmark_location |
#   facility_location | distress_navigation | multi_hop_navigation |
#   nearby_context
# ═════════════════════════════════════════════════════════════════════════════

print("\n[2/5] Building floor_plan_navigation…")

col_nav = client.create_collection("floor_plan_navigation", embedding_function=ef)

nav_docs, nav_metas, nav_ids = [], [], []

# — Entrances ----------------------------------------------------------------
for ent in fp["entrances"]:
    near = ", ".join(ent.get("near_halls", []))
    doc = (
        f"{ent['name_en']} (German: {ent.get('name_de', '')}) "
        f"is an entrance to {venue_info['name']}. "
        f"It is near halls: {near}."
    )
    nav_docs.append(doc)
    nav_metas.append({
        "intent":      "booth_navigation|hall_navigation|facility_location|distress_navigation|multi_hop_navigation",
        "json_source": "Floor Plan",
        "entity_type": "entrance",
        "entrance_id": s(ent["id"]),
        "near_halls":  near,
    })
    nav_ids.append(f"entrance_{ent['id']}")

# — Gates --------------------------------------------------------------------
for gate in fp["gates"]:
    doc = (
        f"{gate['label_en']} (German: {gate['label_de']}) "
        f"is a gate at {venue_info['name']}."
    )
    nav_docs.append(doc)
    nav_metas.append({
        "intent":      "booth_navigation|facility_location",
        "json_source": "Floor Plan",
        "entity_type": "gate",
        "gate_id":     s(gate["id"]),
    })
    nav_ids.append(f"gate_{gate['id']}")

# — Structures / Buildings ---------------------------------------------------
for st in fp["structures"]:
    subs     = st.get("sub_areas") or []
    sub_text = f" Sub-areas: {', '.join(subs)}." if subs else ""
    par_text = f" Part of {st['parent']}." if st.get("parent") else ""
    doc = (
        f"{st['label']} is a {st['type'].replace('_', ' ')} "
        f"at {venue_info['name']}.{sub_text}{par_text}"
    )
    nav_docs.append(doc)
    nav_metas.append({
        "intent":         "hall_navigation|landmark_location|nearby_context",
        "json_source":    "Floor Plan",
        "entity_type":    "structure",
        "structure_id":   s(st["id"]),
        "structure_type": s(st["type"]),
    })
    nav_ids.append(f"structure_{st['id']}")

# — Spatial relationships ----------------------------------------------------
for rel in fp["relationships"]:
    doc = (
        f"At {venue_info['name']}: "
        f"{rel['from']} {rel['type'].replace('_', ' ')} {rel['to']} "
        f"(confidence: {s(rel.get('confidence', 'unknown'))})."
    )
    nav_docs.append(doc)
    nav_metas.append({
        "intent":        "landmark_location|nearby_context|hall_navigation",
        "json_source":   "Floor Plan",
        "entity_type":   "relationship",
        "from_entity":   s(rel["from"]),
        "to_entity":     s(rel["to"]),
        "relation_type": s(rel["type"]),
    })
    nav_ids.append(stable_id("rel", f"{rel['from']}|{rel['to']}|{rel['type']}"))

# — Roads & Streets ----------------------------------------------------------
for road in fp["roads_and_streets"]:
    doc = f"{road} is a road or street near {venue_info['name']}, Berlin."
    nav_docs.append(doc)
    nav_metas.append({
        "intent":      "facility_location|booth_navigation|multi_hop_navigation",
        "json_source": "Floor Plan",
        "entity_type": "road",
    })
    nav_ids.append(stable_id("road", road))

col_nav.add(documents=nav_docs, metadatas=nav_metas, ids=nav_ids)
print(f"    ✓ {col_nav.count()} documents added")


# ═════════════════════════════════════════════════════════════════════════════
# COLLECTION 3 — floor_plan_amenities
#
# What it stores:
#   Food & catering points, lounges & VIP areas, press & conference
#   facilities, and service/logistics locations.
#
# Covers intents:
#   facility_location | booth_zone | landmark_location
# ═════════════════════════════════════════════════════════════════════════════

print("\n[3/5] Building floor_plan_amenities…")

col_amen = client.create_collection("floor_plan_amenities", embedding_function=ef)

amen_docs, amen_metas, amen_ids = [], [], []

AMENITY_CATEGORY_MAP = {
    "food_and_catering_points": ("food_and_catering", "facility_location|booth_zone"),
    "lounges_and_vip":          ("lounge_vip",         "facility_location|landmark_location"),
    "press_and_conference":     ("press_conference",   "facility_location|landmark_location"),
    "service_and_logistics":    ("service_logistics",  "facility_location"),
}

for section, (etype, intent) in AMENITY_CATEGORY_MAP.items():
    for item in fp["amenities_and_services"].get(section, []):
        loc_note = item.get("location_note", "")
        loc_text = f" Located at: {loc_note}." if loc_note else ""
        doc = (
            f"{item['label']} is a {etype.replace('_', ' ')} facility "
            f"at {venue_info['name']}.{loc_text}"
        )
        amen_docs.append(doc)
        amen_metas.append({
            "intent":         intent,
            "json_source":    "Floor Plan",
            "entity_type":    etype,
            "facility_id":    s(item["id"]),
            "facility_label": s(item["label"]),
        })
        amen_ids.append(f"amenity_{item['id']}")

col_amen.add(documents=amen_docs, metadatas=amen_metas, ids=amen_ids)
print(f"    ✓ {col_amen.count()} documents added")


# ═════════════════════════════════════════════════════════════════════════════
# COLLECTION 4 — floor_plan_transport
#
# What it stores:
#   Public transport stops (U-Bahn, S-Bahn, Bus), parking areas,
#   external shuttle lines, fairground shuttles, taxi/bus areas.
#
# Covers intents:
#   facility_location | multi_hop_navigation | booth_navigation
# ═════════════════════════════════════════════════════════════════════════════

print("\n[4/5] Building floor_plan_transport…")

col_transport = client.create_collection("floor_plan_transport", embedding_function=ef)

trans_docs, trans_metas, trans_ids = [], [], []

# — Public transport stops ---------------------------------------------------
for pt in fp["public_transport"]:
    modes = ", ".join(pt.get("modes", []) or [])
    doc = (
        f"{pt['name']} is a public transport stop near {venue_info['name']}. "
        f"Transport modes available: {modes}."
    )
    trans_docs.append(doc)
    trans_metas.append({
        "intent":      "facility_location|multi_hop_navigation|booth_navigation",
        "json_source": "Floor Plan",
        "entity_type": "public_transport",
        "stop_id":     s(pt["id"]),
        "modes":       modes,
    })
    trans_ids.append(f"pt_{pt['id']}")

# — Parking areas ------------------------------------------------------------
for p in fp["parking"]:
    doc = f"Parking area {p['id']} is available near {venue_info['name']}."
    trans_docs.append(doc)
    trans_metas.append({
        "intent":      "facility_location",
        "json_source": "Floor Plan",
        "entity_type": "parking",
        "parking_id":  s(p["id"]),
    })
    trans_ids.append(f"parking_{p['id']}")

# — External shuttle lines (fields: code, route_en) --------------------------
for i, line in enumerate(fp["shuttles"].get("external_shuttle_lines", []) or []):
    code  = s(line.get("code",     f"ext_{i}"))
    route = s(line.get("route_en", line.get("route_de", "")))
    doc   = (
        f"External shuttle line {code}: {route}. "
        f"Departs from Messe Berlin / {venue_info['name']}."
    )
    trans_docs.append(doc)
    trans_metas.append({
        "intent":      "facility_location|multi_hop_navigation",
        "json_source": "Floor Plan",
        "entity_type": "shuttle_external",
        "shuttle_id":  code,
        "route":       route,
    })
    trans_ids.append(f"shuttle_ext_{code}")

# — Fairground shuttles (fields: id, route_en, from, to) ---------------------
for line in fp["shuttles"].get("fairground_shuttles", []) or []:
    sid   = s(line.get("id",       ""))
    route = s(line.get("route_en", line.get("route_de", "")))
    frm   = s(line.get("from",     ""))
    to    = s(line.get("to",       ""))
    doc   = (
        f"Fairground shuttle {sid}: {route}. "
        f"Connects {frm} to {to} within {venue_info['name']}."
    )
    trans_docs.append(doc)
    trans_metas.append({
        "intent":      "facility_location|multi_hop_navigation",
        "json_source": "Floor Plan",
        "entity_type": "shuttle_fairground",
        "shuttle_id":  sid,
        "route":       route,
        "from":        frm,
        "to":          to,
    })
    trans_ids.append(f"shuttle_fg_{sid}")

# — Taxi / bus summary -------------------------------------------------------
approx = fp["taxi_and_bus_points"].get("approximate_areas", []) or []
if approx:
    doc = (
        f"Taxi and bus pickup/drop-off areas at {venue_info['name']}: "
        + "; ".join(str(a) for a in approx) + "."
    )
    trans_docs.append(doc)
    trans_metas.append({
        "intent":      "facility_location|multi_hop_navigation",
        "json_source": "Floor Plan",
        "entity_type": "taxi_bus_areas",
    })
    trans_ids.append("taxi_bus_areas")

col_transport.add(documents=trans_docs, metadatas=trans_metas, ids=trans_ids)
print(f"    ✓ {col_transport.count()} documents added")


# ═════════════════════════════════════════════════════════════════════════════
# COLLECTION 5 — exhibitors
#
# What it stores:
#   One document per exhibitor (2 665 total).
#   Includes name, aliases, hall/stand location(s), country, city,
#   and product categories.
#
# Covers intents:
#   booth_location | exhibitor_discovery | exhibitor_listing |
#   product_info | nearby_context
# ═════════════════════════════════════════════════════════════════════════════

print("\n[5/5] Building exhibitors (2 665 records, batched)…")

col_ex = client.create_collection("exhibitors", embedding_function=ef)

BATCH_SIZE = 200          # flush to ChromaDB every N records
ex_docs, ex_metas, ex_ids = [], [], []
processed = 0

for exhibitor in exhibitors:
    eid        = s(exhibitor["id"])
    name       = s(exhibitor["name"])
    aliases    = exhibitor.get("name_aliases", []) or []
    locations  = exhibitor.get("locations",    []) or []
    address    = exhibitor.get("address",      {}) or {}
    country    = s(exhibitor.get("country",    ""))
    categories = exhibitor.get("categories",   []) or []

    # Build human-readable location string
    loc_parts = []
    for loc in locations:
        if loc.get("type") == "hall":
            loc_parts.append(f"Hall {loc['hall']}, Stand {loc['stand']}")
        elif loc.get("raw"):
            loc_parts.append(loc["raw"])
    loc_text = "; ".join(loc_parts) if loc_parts else "location not listed"

    alias_text = f" Also known as: {', '.join(aliases)}." if aliases else ""
    cat_text   = f" Product categories: {', '.join(categories)}." if categories else ""
    city       = s(address.get("city", ""))
    city_text  = (
        f" Based in {city}, {country}." if city
        else (f" Based in {country}." if country else "")
    )

    doc = (
        f"{name} is exhibiting at {event_info['name']} "
        f"({event_info['dates_display']}, Berlin, Germany). "
        f"Location: {loc_text}.{alias_text}{cat_text}{city_text}"
    )

    # Primary hall/stand (first location entry) for fast metadata filtering
    ph = s(locations[0].get("hall",  "")) if locations else ""
    ps = s(locations[0].get("stand", "")) if locations else ""

    ex_docs.append(doc)
    ex_metas.append({
        "intent":         "booth_location|exhibitor_discovery|exhibitor_listing|product_info|nearby_context",
        "json_source":    "Exhibition List",
        "entity_type":    "exhibitor",
        "exhibitor_id":   eid,
        "name":           name,
        "country":        country,
        "hall":           ph,
        "stand":          ps,
        "categories":     ", ".join(categories) if categories else "",
        "multi_location": "true" if len(locations) > 1 else "false",
    })
    ex_ids.append(f"exhibitor_{eid}")

    if len(ex_docs) >= BATCH_SIZE:
        col_ex.add(documents=ex_docs, metadatas=ex_metas, ids=ex_ids)
        processed += len(ex_docs)
        ex_docs, ex_metas, ex_ids = [], [], []
        print(f"    … {processed} / {len(exhibitors)} flushed", end="\r")

# Final flush
if ex_docs:
    col_ex.add(documents=ex_docs, metadatas=ex_metas, ids=ex_ids)
    processed += len(ex_docs)

print(f"    ✓ {col_ex.count()} documents added        ")


# ── CELL 8: Summary ──────────────────────────────────────────────────────────

COLLECTION_NAMES = [
    "floor_plan_halls",
    "floor_plan_navigation",
    "floor_plan_amenities",
    "floor_plan_transport",
    "exhibitors",
]

total = sum(
    client.get_collection(c, embedding_function=ef).count()
    for c in COLLECTION_NAMES
)

print("\n" + "═" * 58)
print("  ChromaDB build complete")
print("─" * 58)
for c in COLLECTION_NAMES:
    col = client.get_collection(c, embedding_function=ef)
    print(f"  {c:<30} {col.count():>5} docs")
print("─" * 58)
print(f"  Total                          {total:>5} docs")
print(f"  Location: {DB_PATH}")
print("═" * 58)


# ── CELL 9: Smoke tests ───────────────────────────────────────────────────────
# Quick sanity-check queries — verify retrieval works end-to-end.

print("\n── Smoke tests ──────────────────────────────────────────────────────")

tests = [
    (col_halls,     "Where is Hall 7 located?"),
    (col_nav,       "Which entrance is closest to Hall 21?"),
    (col_amen,      "Where can I get food or a coffee?"),
    (col_transport, "How do I get to the venue by U-Bahn?"),
    (col_ex,        "Where is the Scheidt & Bachmann stand?"),
]

for col, query in tests:
    res = col.query(query_texts=[query], n_results=1)
    doc = res["documents"][0][0]
    meta = res["metadatas"][0][0]
    print(f"\nQ: {query}")
    print(f"   → {doc[:110]}{'…' if len(doc) > 110 else ''}")
    print(f"   [entity_type={meta['entity_type']}  intent={meta['intent'][:50]}…]")




Loading all-MiniLM-L6-v2 (downloads ~90 MB on first run)…


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready.
Loading floor plan JSON…
Loading exhibitors JSON…
Event  : InnoTrans 2026  (22–25 September 2026)
Venue  : Berlin Expo Center
Halls  : 43
Exhibitors: 2665
ChromaDB initialised at /content/chromadb_innotrans

[1/5] Building floor_plan_halls…
    ✓ 43 documents added

[2/5] Building floor_plan_navigation…
    ✓ 62 documents added

[3/5] Building floor_plan_amenities…
    ✓ 17 documents added

[4/5] Building floor_plan_transport…
    ✓ 29 documents added

[5/5] Building exhibitors (2 665 records, batched)…
    ✓ 2665 documents added        

══════════════════════════════════════════════════════════
  ChromaDB build complete
──────────────────────────────────────────────────────────
  floor_plan_halls                  43 docs
  floor_plan_navigation             62 docs
  floor_plan_amenities              17 docs
  floor_plan_transport              29 docs
  exhibitors                      2665 docs
──────────────────────────────────────────────────────────
  Total  

In [17]:
# ── CELL 10: Download the database (Colab only) ───────────────────────────────

import shutil
shutil.make_archive("/content/chromadb_innotrans", "zip", "/content/chromadb_innotrans")
print("Archive created: /content/chromadb_innotrans.zip")
#
from google.colab import files
files.download("/content/chromadb_innotrans.zip")


Archive created: /content/chromadb_innotrans.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>